# Chemical kinetics: solution map and parameter estimation

This notebook accompanies the Chapter 8 exercises on chemical kinetics. We first compute trajectories with `scipy.integrate.odeint`. These trajectories are used to learn the discrete solution map

\[
S_h:q\longmapsto \bigl(u(t_j;q)\bigr)_{0\leq j\leq N_t}.
\]

Once the machine is trained, we differentiate it with respect to the kinetic parameters $q$ to solve an inverse problem, then precondition this problem using sensitivities.

## Outline

1. [Reactions and integration with `odeint`](#odeint-cinetique)
2. [Data for the solution map](#donnees-cinetique)
3. [A mass-conserving machine](#machine-cinetique)
4. [Solution-map error](#erreur-cinetique)
5. [Observations and inverse problem](#inverse-cinetique)
6. [Validation with a new solve](#validation-cinetique)
7. [Sensitivity-based preconditioning](#preconditionnement-cinetique)

In [ ]:
from time import perf_counter

import jax
import jax.numpy as jnp
import flax
from flax import nnx
import optax
import matplotlib.pyplot as plt
import numpy as np
import scipy
from scipy.integrate import odeint

jax.config.update("jax_enable_x64", True)
print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, SciPy {scipy.__version__}"
)

<a id="odeint-cinetique"></a>
## 1. Reactions and integration with `odeint`

Consider

\[
A\xrightarrow{q_1}B\xrightarrow{q_2}C,
\qquad
(a(0),b(0),c(0))=(1,0,0),
\]

which gives

\[
a'=-q_1a,\qquad b'=q_1a-q_2b,\qquad c'=q_2b.
\]

The sum $a+b+c$ is conserved. Write the right-hand side using the convention `f(t, u, ...)`, then use the `tfirst=True` option of [`odeint`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.odeint.html).

In [ ]:
# Complete this section.

<a id="donnees-cinetique"></a>
## 2. Data for the solution map

Sample $q$ in the rectangle $\Theta=[q_{1,\min},q_{1,\max}]\times[q_{2,\min},q_{2,\max}]$. For each parameter, `odeint` produces an output of shape `N_TEMPS × 3`. The data are therefore pairs

\[
(q_i,S_h(q_i))\in\mathbb R^2\times\mathbb R^{N_t\times3}.
\]

Separate training, validation and test data during construction.

In [ ]:
# Complete this section.

<a id="machine-cinetique"></a>
## 3. A mass-conserving machine

The machine receives only $q\in\mathbb R^2$ and produces the entire discrete trajectory. Its final layer computes $3(N_t-1)$ numbers, reorganises them by time and applies `softmax` to the three concentrations. Thus, for $t_j>0$, the outputs are positive and sum to one. The initial value $(1,0,0)$ is added exactly.

This structure is not learned: it is imposed by the definition of the machine.

In [ ]:
# Complete this section.

In [ ]:
# Complete this section.

<a id="erreur-cinetique"></a>
## 4. Solution-map error

Compute discrete $L^2$ and $L^\infty$ errors for each test parameter. Check positivity, the initial condition and conservation separately. Plot some trajectories not used for training.

In [ ]:
# Complete this section.

<a id="inverse-cinetique"></a>
## 5. Observations and inverse problem

Choose a parameter $q_*$ unknown to the algorithm and observe only $b$ and $c$ at six times. Add small constructed perturbations. Then seek

\[
\widehat q\in\operatorname*{argmin}_{q\in\Theta}
\frac1{|I|}\sum_{j\in I}
\left\|P\Phi(q,p)_j-z_j\right\|^2,
\qquad P(a,b,c)=(b,c).
\]

To enforce $q\in\Theta$, set $q(r)=q_{\min}+(q_{\max}-q_{\min})\operatorname{sigmoid}(r)$ and optimise $r\in\mathbb R^2$.

In [ ]:
# Complete this section.

In [ ]:
Q_MIN_JAX = jnp.asarray(Q_MIN)
Q_MAX_JAX = jnp.asarray(Q_MAX)
Q_CENTRE_JAX = jnp.asarray(Q_CENTRE)
Q_DEMI_LARGEUR_JAX = jnp.asarray(Q_DEMI_LARGEUR)
INDICES_OBS_JAX = jnp.asarray(indices_observations)
OBSERVATIONS_JAX = jnp.asarray(observations)


def q_depuis_r(r):
    return Q_MIN_JAX + (Q_MAX_JAX - Q_MIN_JAX) * jax.nn.sigmoid(r)


def objectif_inverse(machine, r):
    q = q_depuis_r(r)
    q_n = ((q - Q_CENTRE_JAX) / Q_DEMI_LARGEUR_JAX).reshape(1, 2)
    prediction = machine(q_n)[0, INDICES_OBS_JAX, 1:]
    return jnp.mean((prediction - OBSERVATIONS_JAX) ** 2)


@nnx.jit
def valeur_gradient_inverse(machine, r):
    return jax.value_and_grad(objectif_inverse, argnums=1)(machine, r)

In [ ]:
# Complete this section.

<a id="validation-cinetique"></a>
## 6. Validation with a new solve

The preceding optimisation called only the machine. Now recompute **a single trajectory** with `odeint` at the estimated parameter. Compare:

- the observations;
- the machine at the estimated parameter;
- `odeint` at the estimated parameter;
- `odeint` at the exact parameter, available only in this constructed experiment.

A small value of $\mathcal J_p$ is not sufficient: the inverse problem inherits the learned operator's error.

In [ ]:
# Complete this section.

<a id="preconditionnement-cinetique"></a>
## 7. Sensitivity-based preconditioning

Return to the physical coordinates $q$. If

\[
r(q)=\frac{\mathcal O_p(q)-z}{\sqrt{2|I|}},
\qquad
J_p(q)=D_qr(q),
\]

then $\mathcal J_p(q)=\tfrac12\|r(q)\|^2$ and

\[
\nabla\mathcal J_p(q)=J_p(q)^\top r(q),
\qquad
G_p(q)=J_p(q)^\top J_p(q).
\]

Compute the Jacobian by automatic differentiation. Compare the Euclidean gradient with the damped Gauss--Newton direction. The eigenvalues of $G_p$ measure which parameter combinations are visible in the observations.

In [ ]:
# Complete this section.

In [ ]:
# Complete this section.

## Summary

The learned solution map is a mathematical machine whose input is the kinetic parameter and whose output is an entire trajectory. Its output structure imposes positivity, the initial condition and conservation. The inverse problem then reveals three distinct effects: machine approximation, observations and optimisation.

Machine sensitivities finally provide a natural metric on the parameters. In this setting, Gauss--Newton, the natural gradient and preconditioning describe the same calculation from three perspectives.